In [1]:
import requests
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import chromedriver_autoinstaller

CLIENT_ID = "CLIENT_ID"
CLIENT_SECRET = "SECRET"

query = 'XT-6'

headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}


blog_links = []
titles = []

for start in range(1, 1001, 100): 
    url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"
    res = requests.get(url, headers=headers)
    items = res.json().get("items", [])

    for item in items:
        link = item['link']
        if "blog.naver.com" in link:
            blog_links.append(link)
            titles.append(item['title'])
    
    time.sleep(5)

In [2]:
print(f"총 수집된 블로그 링크 수: {len(blog_links)}")

총 수집된 블로그 링크 수: 985


코드 목적 : 블로그 리뷰 내용을 csv로 저장하기 위함


df_result : api로 수집한 제목, url, 내용 + 수집하는 과정 진행 및 html태그 제거한 csv


blog_review : df_result에서 내용만 저장한 csv

In [ ]:
# 출력 결과(블로그 주소)는 숨김처리하였습니다.
chromedriver_autoinstaller.install()
options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

contents = []

for i, link in enumerate(blog_links):
    if i % 100 == 0:
        if i != 0:
            driver.quit()
            time.sleep(2)
        driver = webdriver.Chrome(options=options)
    
    try:
        print(f"[{i+1}/{len(blog_links)}] 수집 중: {link}")
        driver.get(link)
        time.sleep(2.5)
        driver.switch_to.frame("mainFrame")
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        content = soup.select_one("div.se-main-container")
        if content:
            contents.append(content.get_text(strip=True))
        else:
            content_old = soup.select_one("div#postViewArea")
            if content_old:
                contents.append(content_old.get_text(strip=True))
            else:
                contents.append("본문을 찾을 수 없습니다.")
    except Exception as e:
        contents.append(f"오류 발생: {e}")
        print(f"❌ 오류 발생: {e}")

driver.quit()

In [4]:
errors = [c for c in contents if "오류 발생" in c]
print(f"오류 발생 건수: {len(errors)}")
for e in errors[:5]:
    print(e)

오류 발생 건수: 0


In [5]:
import pandas as pd

df_result = pd.DataFrame({
    '제목': titles,
    '링크': blog_links,
    '내용': contents
})

print(df_result.head())

print("\n✅ 총 수집된 행 수:", len(df_result))
print("✅ 본문 수집 성공:", df_result[~df_result['내용'].str.contains("오류 발생", na=False)].shape[0])
print("❌ 본문 수집 실패:", df_result['내용'].str.contains("오류 발생", na=False).sum())

                                              제목  \
,0      쿠팡에서 구매한 아디다스 운동화 <b>오즈위고</b> - 트리플블랙...    
,1        아디다스 <b>오즈위고</b> 발볼 넓은 운동화 사이즈 팁 받고 구입!   
,2         국내매장정품 아디다스 <b>오즈위고</b> GW8016을 경험해보세요   
,3   아디다스 <b>오즈위고</b> 프로 - 블랙 IH0397 발등 높은 칼발...    
,4  쿠팡에서 구매한 아디다스 <b>오즈위고</b> EE6999로 시작된 일상의...    
,
,                                                  링크  \
,0   https://blog.naver.com/ueqljzr91vo2/223874836802   
,1      https://blog.naver.com/skwlsdldi/223496176978   
,2  https://blog.naver.com/eq5m3em0dl1xv6/22382067...   
,3     https://blog.naver.com/be_happy23/223857117334   
,4   https://blog.naver.com/u9bttaa1zfv3/223874852424   
,
,                                                  내용  
,0  ​이포스팅은쿠팡파트너스활동의일환으로이에따른일정액의수수료를제공받습니다.출처-네이버이미...  
,1  요즘 걷고 뛰는거에 관심이 많은우리 부부 :)편한게 최고라서 신발을 구입할 때에도 ...  
,2  ​이포스팅은쿠팡파트너스활동의일환으로,이에따른일정액의수수료를제공받습니다.출처-네이버이...  
,3  운동화 리뷰는 처음이라 떨림다음에신발 구입할 때삽질을 최소화 하기 위한기록용 리뷰!...  
,4  ​이포스팅은쿠팡파트너스활동의일환으로이에따른일정액의수수료를제공받습니다.출처-네이버이미...  
,

In [6]:
import pandas as pd
import re

def remove_html_tags(text):
    if pd.isna(text): 
        return ""
    return re.sub('<.*?>', '', text)

df_result['제목'] = df_result['제목'].apply(remove_html_tags)
df_result['내용'] = df_result['내용'].apply(remove_html_tags)

df_result.to_csv("네이버블로그리뷰(살로몬).csv", index=True, encoding='utf-8-sig')
print("✅ HTML 태그 제거 후 저장 완료")

✅ HTML 태그 제거 후 저장 완료
